# Milestones 1 & 2 — Naive loop and KV cache

Thin driver. All logic lives in `engine/`.

In [ ]:
# Colab only — skip if running locally in the repo.
!git clone https://github.com/YOURNAME/tiny-inference-engine.git
%cd tiny-inference-engine
!pip install -q -e .

In [ ]:
from engine import load
from engine.naive import measure_naive, warmup
from engine.cached import check_matches_naive, measure_cached
from engine.prompts import SHORT, LONG
from engine.results import report, save

rt = load()
rt.describe()

In [ ]:
MAX_NEW = 400
warmup(rt, SHORT)
print("warmup done")

## Milestone 1 — baseline

In [ ]:
m1 = [
    report(measure_naive(rt, "short_prompt", SHORT, MAX_NEW)),
    report(measure_naive(rt, "long_prompt", LONG, MAX_NEW)),
]
save(m1)

Flat while short (launch-bound), then linear growth. Linear per step means
quadratic in total — that slope is the missing cache.

## Milestone 2 — KV cache

In [ ]:
assert check_matches_naive(rt, SHORT), "cache diverged"
assert check_matches_naive(rt, LONG), "cache diverged"

In [ ]:
m2 = [
    report(measure_cached(rt, "short_prompt", SHORT, MAX_NEW)),
    report(measure_cached(rt, "long_prompt", LONG, MAX_NEW)),
]
save(m2)

## Comparison

In [ ]:
header = f"{'':<14}{'naive tok/s':>13}{'cached tok/s':>14}{'speedup':>10}"
print(header)
print("-" * len(header))
for a, b in zip(m1, m2):
    print(f"{a['label']:<14}{a['tokens_per_sec']:>13}{b['tokens_per_sec']:>14}"
          f"{b['tokens_per_sec'] / a['tokens_per_sec']:>9.1f}x")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, a, b in zip(axes, m1, m2):
    ax.plot(range(0, len(a["curve"]) * 16, 16), a["curve"], label="m1 — no cache")
    ax.plot(range(0, len(b["curve"]) * 16, 16), b["curve"], label="m2 — KV cache")
    ax.set_title(f"{a['label']} ({a['prompt_tokens']} prompt tokens)")
    ax.set_xlabel("generation step")
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("step latency (ms)")
plt.tight_layout()
plt.savefig("benchmarks/plots/m1_vs_m2.png", dpi=140)
plt.show()